# nano Beatrix v1 — the output layer, screened before the train

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_v1_head_decay.ipynb)

**Where the first stage left it.** The first stage (`nano_v1_lastfix_screen.ipynb`) bounded the last block's write three ways on Castor (4,960 steps from the v1 start, the 2s recipe):

| arm | share of the last block's output on its one direction | after the final norm | rank of the spread around it (of 256) | bits a byte at 4,960 |
|---|---|---|---|---|
| plain (seed 3) | 0.928 | 0.901 | 11 | 1.3588 |
| register | 0.888 | 0.871 | 46 | 1.3556 |
| temperature | 0.873 | 0.852 | 43 | 1.3581 |
| gain | 0.921 | 0.899 | 15 | 1.3596 |

The bounds widened the spread around the direction but barely held the direction itself: the earlier blocks carried it instead (block 5's output 0.80 on it under the register and the temperature, 0.58 in the plain run).

**What the direction does** (read on the plain run's saved checkpoints, 16 held-out documents). The final norm rescales every byte to one size, so the room the direction takes in a byte is room the content loses: the direction's size per byte sets how confident the prediction is.

| change to the direction just before the final norm (plain run, step 4,960) | prediction entropy (bits) | bits a byte |
|---|---|---|
| none | 1.36 | 1.357 |
| removed | 0.41 | 6.32 |
| halved | 0.89 | 1.82 |
| doubled | 2.00 | 1.70 |
| held at its mean size for every byte | 1.33 | 1.461 |

The bytes that carry more of it predict with more uncertainty (rank correlation 0.79 between a byte's share on it and its prediction's entropy, at every checkpoint read).

**Why its share keeps climbing: the output layer keeps growing.**

| plain run, step | the readout's size | its gain on the content | the content's room after the norm | the content's net loudness | share on the direction |
|---|---|---|---|---|---|
| 1,145 | 32.9 | 3.28 | 0.51 | 1.67 | 0.813 |
| 2,000 | 40.4 | 4.10 | 0.41 | 1.67 | 0.876 |
| 4,000 | 55.4 | 5.44 | 0.33 | 1.79 | 0.921 |
| 4,960 | 60.8 | 5.83 | 0.32 | 1.84 | 0.928 |

The readout reads the content 78% louder by the end, the direction takes 38% of the content's room, and the net loudness rises a tenth: the direction holds the predictions' confidence against a growing readout. The readout (`head.w_h`, `head.w_s`) trains under Muon, which takes a full-size step every update and has no decay. Muon's reference guidance trains classifier heads with AdamW ([github.com/KellerJordan/Muon](https://github.com/KellerJordan/Muon)), and Muon-trained weights without decay keep growing (Liu et al. 2025, [arXiv 2502.16982](https://arxiv.org/abs/2502.16982), s.2.2).

**The arms** (the plain Castor start, seed 3; the same data in the same order as the first stage; the 2s recipe except the output layer's two readout matrices; 4,960 steps):

| arm | the output layer's readout | decoupled weight decay |
|---|---|---|
| `c-head-adam` | AdamW at decay 0 (the Adam update) | 0 |
| `c-head-adamw0.1` | AdamW | 0.1 (the common setting) |
| `c-head-adamw1.2` | AdamW | 1.2 (0.1's total pull over the full 61,422-step diet, within these 4,960 steps) |

The readout's learning rate is the recipe's Adam rate (0.0003); its biases and every other parameter keep the 2s recipe (Muon on the matrices, Adam without decay on the rest). The plain references are the first stage's `c-bare-s3` (the same start) and `c-bare-s4` (the seed bar), read from the repo.

**What is read** every 500 steps and at both phase ends: everything the first stage reads (bits a byte, the share on the last block's direction at every block and after the final norm, the last bank's write, every block's rank, the final norm's gain and bias), and the output layer's size and loudness (TensorBoard `lastwrite/head_*`). At the end: the drift memory on random bytes (the exit's H).

**How an arm passes** (written before any arm runs): (i) the share on the last block's direction stops climbing after the warmup (at 4,960 no more than 0.01 above its value at 1,500) and sits under the plain run's at every read from 1,500 on; (ii) bits a byte at 4,960 within max(0.001, the two plain seeds' spread) of `c-bare-s3`; (iii) the exit's H inside the plain runs' spread. The readout's size and loudness show whether an arm acts the way the cause read says it should (the readout stops growing, then the direction stops climbing).

**Length:** about 26 minutes an arm on an RTX PRO 6000, about 80 minutes for the three; the cell can be stopped and rerun (each arm resumes from the repo, finished arms are read back).

**Token:** `HF_TOKEN` with write access to AbstractPhil/alephllm-nano-beatrix-training (the records land under `v1_lastfix/<arm>/`, beside the first stage's).

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if the version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.29"
import torch
import geolip.nano_beatrix as N
import geolip.alephllm as A
from geolip.nano_beatrix import lastfix_screen as LS
print('geolip.nano_beatrix', N.__version__, '| geolip.alephllm', A.__version__, '| torch', torch.__version__,
      '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 29), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart, then rerun from C1')
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()                                   # the value is never printed
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN

# ============ THE DECISION BLOCK ============
OUT = '/content/nano_v1_head_decay'                   # the session folder: the starts, the runs, the records
REPO = LS.REPO                                        # AbstractPhil/alephllm-nano-beatrix-training, under v1_lastfix/<arm>/
NAMES = ['c-head-adam', 'c-head-adamw0.1', 'c-head-adamw1.2']
REFS = ['c-bare-s3', 'c-bare-s4']                     # the first stage's plain runs (read from the repo, never retrained here)
MAX_HOURS = None                                      # this session's budget in hours (None: until every arm is done)
print('arms:', ', '.join(NAMES), '| ships to', REPO if HF_TOKEN else 'nothing (tokenless)')

In [ ]:
# C2 (P) — PREFLIGHT: every arm's start (checked against a fresh build), its optimizer split, the length, the push probe. Trains nothing.
import json, time
from geolip.alephllm.model.alephlm import AlephLM
from geolip.alephllm.train.optim import build_optimizers
births = {n: LS.write_arm_birth(LS.BY_NAME[n], f'{OUT}/births') for n in NAMES}
for n in NAMES:
    t = LS.arm_preset(LS.BY_NAME[n], births[n]).train
    m = AlephLM(LS.arm_cfg(LS.BY_NAME[n]))
    opts = build_optimizers(m, t.muon_lr, t.muon_momentum, t.adam_lr, decay_prefixes=t.decay_params, decay_wd=t.decay_wd)
    g = opts[1].param_groups[1]
    print(f"{n}: {sum(p.numel() for p in g['params']):,} readout weights on {type(opts[1]).__name__} at decay {g['weight_decay']} "
          f"(lr {t.adam_lr}); {sum(p.numel() for p in opts[0].param_groups[0]['params']):,} on Muon")
mins = 3 * (4960 * 0.299 + 12 * 25) / 60                                     # the v1 bench on an RTX PRO 6000 at 16 x 4 (2026-10-14)
print(f'-> about {mins:.0f} minutes on an RTX PRO 6000 (training plus a full eval and the gauge every 500 steps)')
PUSH_OK = False
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    path = f'{LS.PREFIX}/preflight/push_probe_head_decay.json'
    api.upload_file(path_or_fileobj=json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
                                                'card': torch.cuda.get_device_name(0), 'arms': NAMES}).encode(),
                    path_in_repo=path, repo_id=REPO, commit_message='v1 head-decay screen: push probe')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: C3 runs locally and ships nothing')

In [ ]:
# C3 (T) — THE SCREEN: arm by arm; the trainer's own lines and the screen's gauge lines ([lastfix]) print as they come. Rerun the cell to
# continue (a new session too): each arm resumes from the repo, finished arms are read back. The stop button saves the running arm first.
RECS = LS.run(OUT, HF_TOKEN if PUSH_OK else None, NAMES, max_hours=MAX_HOURS)
from huggingface_hub import hf_hub_download
from IPython.display import Markdown, display

def with_refs(recs):
    out = {}
    for n in REFS:
        try:
            out[n] = json.load(open(hf_hub_download(REPO, f'{LS.PREFIX}/{n}/screen_record.json', token=HF_TOKEN or False), encoding='utf-8'))
        except Exception as e:  # noqa: BLE001
            print('no record yet for', n, type(e).__name__)
    out.update(recs)
    return out
display(Markdown('**At 1,500 (after the warmup)**\n\n' + LS.table(with_refs(RECS), 1500)))
display(Markdown('**At the end (each arm\'s last read)**\n\n' + LS.table(with_refs(RECS))))

In [ ]:
# C4 — THE CURVES AND THE TABLES from the repo (every session; tokenless: this session's files). Trains nothing.
import json
from IPython.display import Markdown, display
recs, logdir = {}, f'{OUT}/runs'
if HF_TOKEN:
    from huggingface_hub import snapshot_download
    hub = snapshot_download(REPO, token=HF_TOKEN, local_dir=f'{OUT}/hub',
                            allow_patterns=[f'{LS.PREFIX}/{n}/runs/*' for n in REFS + NAMES] + [f'{LS.PREFIX}/{n}/screen_record.json' for n in REFS + NAMES])
    logdir = f'{hub}/{LS.PREFIX}'
    for n in REFS + NAMES:
        p = f'{hub}/{LS.PREFIX}/{n}/screen_record.json'
        if os.path.exists(p):
            recs[n] = json.load(open(p, encoding='utf-8'))
else:
    for n in NAMES:
        p = f'{OUT}/lastfix/{n}.json'
        if os.path.exists(p):
            recs[n] = json.load(open(p, encoding='utf-8'))
display(Markdown('**At 1,500 (after the warmup)**\n\n' + LS.table(recs, 1500)))
display(Markdown('**Each arm\'s last read**\n\n' + LS.table(recs)))
%load_ext tensorboard
%tensorboard --logdir {logdir}

### Reading the result

| read | the plain run | an arm that works |
|---|---|---|
| the readout's size (`lastwrite/head_w_h_fro`) | doubles in 3,800 steps | grows less, or settles |
| share on the last block's direction | climbs from 0.857 at 1,500 to 0.928 | flat or falling after the warmup, under the plain run at every read |
| the content's net loudness (`lastwrite/head_loudness`) | about flat: the direction absorbs the readout's growth | about flat with less of the direction (the room kept instead) |
| bits a byte at 4,960 | 1.3588 (the reference) | within max(0.001, the plain seeds' spread) |
| the exit's H on random bytes | the reference | inside the plain runs' spread |

An arm that passes gives the v1 crafts their output layer's optimizer (one setting in the v1 preset), and the v1 train starts from scratch.

### Changing the screen
| to | the edit in C1 |
|---|---|
| one arm | `NAMES = ['c-head-adamw0.1']` |
| a session budget | `MAX_HOURS = 1` |